In [0]:
--Record that must be rejected 

SELECT
    Member_Name,
    Member_ID,
    Source_Country,
    Enrollment_Date,
    Enrollment_Date_Status,
    CASE
        WHEN Member_Name IS NULL OR TRIM(Member_Name) = ''
            THEN 'MISSING_MEMBER_NAME'
        WHEN Member_ID IS NULL OR TRIM(CAST(Member_ID AS STRING)) = ''
            THEN 'MISSING_MEMBER_ID'
        WHEN Enrollment_Date IS NULL
             OR Enrollment_Date_Status <> 'VALID'
            THEN 'INVALID_ENROLLMENT_DATE'
    END AS Rejection_Reason

FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`

WHERE Member_Name IS NULL
   OR TRIM(Member_Name) = ''
   OR Member_ID IS NULL
   OR TRIM(CAST(Member_ID AS STRING)) = ''
   OR Enrollment_Date IS NULL
   OR Enrollment_Date_Status <> 'VALID';

Member_Name,Member_ID,Source_Country,Enrollment_Date,Enrollment_Date_Status,Rejection_Reason
Jonnathan,2,AUS,null,INVALID,INVALID_ENROLLMENT_DATE


In [0]:
--Create the rejected records Delta table

CREATE OR REPLACE TABLE
delta.`abfss://rejected@incubytedatalake01.dfs.core.windows.net/invalid_member_records/`
USING DELTA
AS
SELECT
    Member_Name,
    Member_ID,
    Source_Country,
    Enrollment_Date,
    Enrollment_Date_Status,
    CASE
        WHEN Member_Name IS NULL OR TRIM(Member_Name) = ''
            THEN 'MISSING_MEMBER_NAME'
        WHEN Member_ID IS NULL OR TRIM(CAST(Member_ID AS STRING)) = ''
            THEN 'MISSING_MEMBER_ID'
        WHEN Enrollment_Date IS NULL
             OR Enrollment_Date_Status <> 'VALID'
            THEN 'INVALID_ENROLLMENT_DATE'
    END AS Rejection_Reason
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
WHERE Member_Name IS NULL
   OR TRIM(Member_Name) = ''
   OR Member_ID IS NULL
   OR TRIM(CAST(Member_ID AS STRING)) = ''
   OR Enrollment_Date IS NULL
   OR Enrollment_Date_Status <> 'VALID';

num_affected_rows,num_inserted_rows


In [0]:
--Verify rejected table
SELECT *
FROM delta.`abfss://rejected@incubytedatalake01.dfs.core.windows.net/invalid_member_records/`;

Member_Name,Member_ID,Source_Country,Enrollment_Date,Enrollment_Date_Status,Rejection_Reason
Jonnathan,2,AUS,null,INVALID,INVALID_ENROLLMENT_DATE


In [0]:
--Invalid DOB records
SELECT
    Member_Name,
    Member_ID,
    Source_Country,
    DOB,
    DOB_Status
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
WHERE DOB_Status = 'INVALID';

Member_Name,Member_ID,Source_Country,DOB,DOB_Status
Mike,1,AUS,null,INVALID


In [0]:
--Save Invalid DOB record
CREATE OR REPLACE TABLE
delta.`abfss://rejected@incubytedatalake01.dfs.core.windows.net/member_data_quality_issues/`
USING DELTA
AS
SELECT
    Member_Name,
    Member_ID,
    Source_Country,
    DOB,
    DOB_Status,
    'INVALID_DOB' AS Issue_Reason
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
WHERE DOB_Status = 'INVALID';

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM delta.`abfss://rejected@incubytedatalake01.dfs.core.windows.net/member_data_quality_issues/`;

Member_Name,Member_ID,Source_Country,DOB,DOB_Status,Issue_Reason
Mike,1,AUS,null,INVALID,INVALID_DOB


In [0]:
--Indetify unmatched redemtion record
CREATE OR REPLACE TABLE
delta.`abfss://rejected@incubytedatalake01.dfs.core.windows.net/unmatched_redemptions/`
USING DELTA
AS
SELECT *
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/member_redemption_join/`
WHERE Member_Name IS NULL;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM delta.`abfss://rejected@incubytedatalake01.dfs.core.windows.net/unmatched_redemptions/`;

member_id,Transaction_ID,Transaction_Date,Partner,Miles_Redeemed,Transaction_Status,Member_Name,Source_Country
223457,RX10091,20240110,AeroLink,12000,COMPLETED,null,null
223457,RX10092,20240113,SkyPoints,5000,PENDING,null,null


In [0]:
--duplicate member names in Gold.

SELECT
    Member_Name,
    COUNT(*) AS Record_Count
FROM (
    SELECT * FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_Australia/`
    UNION ALL
    SELECT * FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_India/`
    UNION ALL
    SELECT * FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_USA/`
)
GROUP BY Member_Name
HAVING COUNT(*) > 1;

Member_Name,Record_Count


In [0]:
--Duplicate redemption teansaction id 
SELECT
    Transaction_ID,
    COUNT(*) AS Record_Count
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/`
GROUP BY Transaction_ID
HAVING COUNT(*) > 1;

Transaction_ID,Record_Count


In [0]:
--ensure there are no zero or negative miles in the redemption data.

SELECT *
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/`
WHERE Miles_Redeemed IS NULL
   OR Miles_Redeemed <= 0;

member_id,feed_date,Transaction_ID,Transaction_Date,Partner,Miles_Redeemed,Transaction_Status
